In [23]:
import os
import io
import base64
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
import torch
import torch.nn as nn
from dash import Dash, dcc, html, Input, Output, State
import plotly.graph_objects as go

In [24]:
# --------------------- MDN 모델 정의 ---------------------
class MDN(nn.Module):
    def __init__(self, out_dim=2, num_mixtures=9):
        super(MDN, self).__init__()
        in_dim = 3

        self.hidden = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
        )
        self.pi = nn.Linear(64, num_mixtures)
        self.mu = nn.Linear(64, num_mixtures * out_dim)
        self.sigma = nn.Linear(64, num_mixtures * out_dim)
        self.num_mixtures = num_mixtures
        self.out_dim = out_dim

    def forward(self, x_tuple):
        x, y, t = x_tuple
        concat = torch.cat([x, y, t], dim=1)
        h = self.hidden(concat)
        pi = torch.softmax(self.pi(h), dim=1)
        mu = self.mu(h).view(-1, self.num_mixtures, self.out_dim)
        sigma = torch.exp(self.sigma(h)).clamp(min=1e-2).view(-1, self.num_mixtures, self.out_dim)
        return pi, mu, sigma

In [25]:
# # --------------------- 시각화 함수 ---------------------
# def generate_mdn_contour(model, x, y, t, device):
#     t_scale = 2400.0
#     coord_scale = 100000.0

#     input_tensor = (
#         torch.tensor([[x / coord_scale]], dtype=torch.float32).to(device),
#         torch.tensor([[y / coord_scale]], dtype=torch.float32).to(device),
#         torch.tensor([[t / t_scale]], dtype=torch.float32).to(device)
#     )

#     model.eval()
#     with torch.no_grad():
#         pi, mu, sigma = model(input_tensor)
#         pi = pi.squeeze().cpu().numpy()
#         mu = mu.squeeze().cpu().numpy()
#         sigma = sigma.squeeze().cpu().numpy()

#     xg = np.linspace(-2, 2, 200)
#     yg = np.linspace(-2, 2, 200)
#     X, Y = np.meshgrid(xg, yg)
#     Z = np.zeros_like(X)

#     for k in range(len(pi)):
#         mux, muy = mu[k]
#         sigx, sigy = sigma[k]
#         px = (1.0 / (np.sqrt(2 * np.pi) * sigx)) * np.exp(-0.5 * ((X - mux) / sigx) ** 2)
#         py = (1.0 / (np.sqrt(2 * np.pi) * sigy)) * np.exp(-0.5 * ((Y - muy) / sigy) ** 2)
#         Z += pi[k] * px * py

#     fig, ax = plt.subplots(figsize=(4, 4), dpi=100)
#     ax.contourf(X, Y, Z, levels=100, cmap="viridis")
#     ax.scatter(mu[:, 0], mu[:, 1], c='red', s=40, edgecolors='white', label='mu')
#     ax.set_title("GMM")
#     ax.set_xlim(-2, 2)
#     ax.set_ylim(-2, 2)
#     ax.axis('off')

#     buf = io.BytesIO()
#     plt.savefig(buf, format='png', bbox_inches='tight')
#     plt.close(fig)
#     buf.seek(0)
#     return base64.b64encode(buf.read()).decode('utf-8')

# # --------------------- 초기 설정 ---------------------
# app = Dash(__name__)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = MDN().to(device)

# def load_latest_model(path="saved_models"):
#     pt_files = [f for f in os.listdir(path) if f.endswith(".pt") and "mdn_epoch_" in f]
#     if not pt_files:
#         raise FileNotFoundError("No .pt model files found in the specified directory.")
#     latest_file = max(pt_files, key=lambda x: int(x.split("mdn_epoch_")[1].split(".pt")[0]))
#     model_path = os.path.join(path, latest_file)
#     print(f"[INFO] Loading model: {model_path}")
#     return torch.load(model_path, map_location=device)

# checkpoint = load_latest_model()
# model.load_state_dict(checkpoint['model_state_dict'])
# model.eval()

# # 이미지 로드
# data_path = "../../../Downloads/archive"
# erangel_img = os.path.join(data_path, 'erangel.jpg')
# with Image.open(erangel_img) as im:
#     buf = io.BytesIO()
#     im.save(buf, format="PNG")
#     encoded_img = base64.b64encode(buf.getvalue()).decode()
#     erangel_base64 = f"data:image/png;base64,{encoded_img}"

# # --------------------- 레이아웃 ---------------------
# app.layout = html.Div([
#     html.H3("ERANGEL MDN Viewer"),
#     dcc.Slider(id="time-slider", min=0, max=2400, step=10, value=2400, 
#                tooltip={"placement": "bottom", "always_visible": True}),
#     html.Div(id='hover-coords', style={"margin": "10px 0"}),
#     html.Div([
#         dcc.Graph(id="map-graph", style={"height": "800px"}),
#         html.Img(id="mdn-overlay", style={"height": "400px", "marginTop": "10px"})
#     ])
# ])

# @app.callback(
#     Output("map-graph", "figure"),
#     Input("time-slider", "value")
# )
# def update_map_figure(time_val):
#     fig = go.Figure()
#     fig.update_layout(
#         xaxis=dict(range=[0, 800000], showgrid=True, tickvals=list(range(0, 800001, 100000))),
#         yaxis=dict(range=[800000, 0], showgrid=True, tickvals=list(range(0, 800001, 100000)), scaleanchor="x"),
#         images=[dict(
#             source=erangel_base64,
#             xref="x", yref="y",
#             x=0, y=0, sizex=800000, sizey=800000,
#             sizing="stretch", opacity=0.8, layer="below"
#         )],
#         margin=dict(t=10, b=10, l=10, r=10),
#         clickmode='event+select'
#     )
#     return fig

# @app.callback(
#     Output("hover-coords", "children"),
#     Output("mdn-overlay", "src"),
#     Input("map-graph", "hoverData"),
#     State("time-slider", "value")
# )
# def show_hover_and_mdn(hoverData, current_time):
#     if hoverData and 'points' in hoverData:
#         pt = hoverData['points'][0]
#         x = pt.get("x")
#         y = pt.get("y")
#         if x is not None and y is not None:
#             img_uri = generate_mdn_contour(model, x, y, current_time, device)
#             return f"Hover Position: x={int(x)}, y={int(y)}", f"data:image/png;base64,{img_uri}"
#     return "Hover Position: -", None

In [39]:
def generate_mdn_contour(model, x, y, t, device):
    t_scale = 2400.0
    coord_scale = 100000.0

    input_tensor = (
        torch.tensor([[x / coord_scale]], dtype=torch.float32).to(device),
        torch.tensor([[y / coord_scale]], dtype=torch.float32).to(device),
        torch.tensor([[t / t_scale]], dtype=torch.float32).to(device)
    )

    model.eval()
    with torch.no_grad():
        pi, mu, sigma = model(input_tensor)
        pi = pi.squeeze().cpu().numpy()
        mu = mu.squeeze().cpu().numpy()
        sigma = sigma.squeeze().cpu().numpy()

    xg = np.linspace(-2, 2, 200)
    yg = np.linspace(-2, 2, 200)
    X, Y = np.meshgrid(xg, yg)
    Z = np.zeros_like(X)

    for k in range(len(pi)):
        mux, muy = mu[k]
        sigx, sigy = sigma[k]
        px = (1.0 / (np.sqrt(2 * np.pi) * sigx)) * np.exp(-0.5 * ((X - mux) / sigx) ** 2)
        py = (1.0 / (np.sqrt(2 * np.pi) * sigy)) * np.exp(-0.5 * ((Y - muy) / sigy) ** 2)
        Z += pi[k] * px * py

    fig, ax = plt.subplots(figsize=(4, 4), dpi=100)
    ax.contourf(X, Y, Z, levels=100, cmap="viridis")
    ax.axis('off')

    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', transparent=True)
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode('utf-8')

# --------------------- 초기 설정 ---------------------
app = Dash(__name__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MDN().to(device)

# 가장 숫자가 큰 모델 로드
def load_latest_model(path="saved_models"):
    pt_files = [f for f in os.listdir(path) if f.endswith(".pt") and "mdn_epoch_" in f]
    if not pt_files:
        raise FileNotFoundError("No .pt model files found in the specified directory.")
    latest_file = max(pt_files, key=lambda x: int(x.split("mdn_epoch_")[1].split(".pt")[0]))
    model_path = os.path.join(path, latest_file)
    print(f"[INFO] Loading model: {model_path}")
    return torch.load(model_path, map_location=device)

checkpoint = load_latest_model()
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# 이미지 로드
data_path = "../../../Downloads/archive"
erangel_img = os.path.join(data_path, 'erangel.jpg')
with Image.open(erangel_img) as im:
    buf = io.BytesIO()
    im.save(buf, format="PNG")
    encoded_img = base64.b64encode(buf.getvalue()).decode()
    erangel_base64 = f"data:image/png;base64,{encoded_img}"

# --------------------- 레이아웃 ---------------------
app.layout = html.Div([
    html.H3("ERANGEL MDN Viewer"),
    dcc.Slider(
        id="time-slider",
        min=0,
        max=2400,
        step=10,
        value=2400,
        marks={i: str(i) for i in range(0, 2401, 300)},
        tooltip={"placement": "bottom", "always_visible": True}
    ),
    html.Div(id='hover-coords', style={"margin": "10px 0"}),
    dcc.Graph(id="map-graph", style={"height": "800px"}, config={"staticPlot": False})
])

@app.callback(
    Output("map-graph", "figure"),
    Input("time-slider", "value"),
    Input("map-graph", "hoverData")
)
def update_map_figure(time_val, hoverData):
    fig = go.Figure()
    fig.update_layout(
        xaxis=dict(range=[0, 800000], showgrid=True, tickvals=list(range(0, 800001, 100000))),
        yaxis=dict(range=[800000, 0], showgrid=True, tickvals=list(range(0, 800001, 100000)), scaleanchor="x"),
        images=[dict(
            source=erangel_base64,
            xref="x", yref="y",
            x=0, y=0, sizex=800000, sizey=800000,
            sizing="stretch", opacity=0.8, layer="below"
        )],
        margin=dict(t=10, b=10, l=10, r=10),
        clickmode='event+select',
        hovermode='closest'
    )

    # 10000 간격 그리드 중점들에 hover용 투명 마커 추가
    grid_x = np.arange(0, 800000, 10000)
    grid_y = np.arange(0, 800000, 10000)
    cx, cy = np.meshgrid(grid_x + 5000, grid_y + 5000)
    fig.add_trace(go.Scatter(
        x=cx.flatten(),
        y=cy.flatten(),
        mode='markers',
        marker=dict(size=8, color='rgba(0,0,0,0.001)'),
        # hoverinfo='x+y',
        hoverinfo='none',
        # hoverinfo='skip',
        hovertemplate='',
        name='hover-capture',
        showlegend=False
    ))

    # MDN 결과 이미지를 오버레이
    if hoverData and 'points' in hoverData and len(hoverData['points']) > 0:
        pt = hoverData['points'][0]
        x_center = pt.get("x")
        y_center = pt.get("y")
        if x_center is not None and y_center is not None:
            mdn_img = generate_mdn_contour(model, x_center, y_center, time_val, device)
            sizex = 100000
            sizey = 100000
            fig.add_layout_image(dict(
                source=f"data:image/png;base64,{mdn_img}",
                xref="x", yref="y",
                x=x_center - sizex / 2,
                y=y_center - sizey / 2,  # ✅ FIXED
                sizex=sizex,
                sizey=sizey,
                sizing="stretch",
                opacity=0.6,
                layer="above"
            ))
    return fig


[INFO] Loading model: saved_models\mdn_epoch_100.pt


C:\Users\darke\AppData\Local\Temp\ipykernel_2692\3254444551.py:53: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.



In [40]:
app.run(debug=True)